# NPM Package: `Refspy`

This is an experimental AI translation of a Python package for indexing biblical references. This page offers a cookbook of useful solutions. It runs in Deno inside Jupyter, allowing Typescript, and is adapted from the Python [NOTEBOOK.ipynb](https://github.com/eukras/refspy/blob/master/NOTEBOOK.ipynb).

## Initialisation


In [1]:
import refsjs from 'refsjs';

const EN = refsjs.refspy();                     //  <-- Refspy defaults to a Protestant canon and en_US localisation,
const FR = refsjs.refspy('catholic', 'fr_FR');  //  <-- French localisation with deuterocanonicals. 

const __ = refsjs.refspy("orthodox");           //  <-- for this demonstration, we'll use English with deuterocanonicals and anagignoskomena.

## Formatting


In [2]:
const r = __.r("Gen 1:1");

console.log([__.name(r), __.book(r), __.abbrevName(r), __.abbrevBook(r), __.numbers(r)]);
console.log([FR.name(r), FR.book(r), FR.abbrevName(r), FR.abbrevBook(r), FR.numbers(r)]);

[ "Genesis 1:1", "Genesis", "Gen 1:1", "Gen", "1:1" ]
[ "Genèse 1, 1", "Genèse", "Gn 1, 1", "Gn", "1, 1" ]


## Find references

Some demonstration text is stored in each language file; we'll use that here.


In [3]:
const text = refsjs.LANGUAGES.en.demonstrationText;

console.log(text);

Human-written Bible references look like Rom 12.1-2, 9-12, 2 Cor. 4:16-5:5,
or Philemon 4-7. They wrap lines, use spaces inconsistently, use colons and
periods interchangeably, indicate abbreviations with periods (or not), and
have commas both between and within references. They are sometimes
malformed, like Mt 1.10000 or 1:3-2, but might also use number
abbreviations like Ps 119:105-12. They might refer to Deuterocanonical
books (DC), like Wis 7:21-30 or 2 Macc 7, or Anagignoskomena (DCO), like 1
Esdras 4:35-40. A book name, like Second Corinthians, may provide context
for references that follow, such as 5:11-15 or vv. 16-21. We want to match,
say, 'John' but not match 'John Smith' in these cases. If we cite Romans
(but then add a reference to 2ndCo 5:11-21 in parentheses), a subsequent
reference like 12:9-21 should still be to Romans. Using letters for partial
verses, as in II Cor 5:11a-15d, has no consistent meaning, so the letters
are ignored. Book aliases that are common words wil

In [4]:
const matches = __.findReferences(text, true, true);  // <-- include books, include nulls

const headers = [
    ["TYPE", "MATCHED", "REFERENCE", "ABBREVIATION"],
];

const data = matches.map(([match, ref]) => {
    if (ref == null) {
        return ["None", match, '--', '--'];
    } else if (ref.isBook()) {
        return ["Book", match, __.name(ref), __.abbrevName(ref)];
    } else {
        return ["Reference", match, __.name(ref), __.abbrevName(ref)];
    }
});

const table = (data) => "<table>" + data.map(row =>
    `<tr>${row.map(cell => `<td>${cell}</td>`).join("")}</tr>`
).join("") + "</table>";

const html = table([...headers, ...data]);

Deno.jupyter.html`${html}`;

TYPE,MATCHED,REFERENCE,ABBREVIATION
Reference,"Rom 12.1-2, 9-12","Romans 12:1–2, 9–12","Rom 12:1–2, 9–12"
Reference,2 Cor. 4:16-5:5,2 Corinthians 4:16–5:5,2 Cor 4:16–5:5
Reference,Philemon 4-7,Philemon 4–7,Phlm 4–7
None,Mt 1.10000,--,--
None,1:3-2,--,--
Reference,Ps 119:105-12,Psalm 119:105–112,Ps 119:105–112
Reference,Wis 7:21-30,Wisdom of Solomon 7:21–30,Wis 7:21–30
Reference,2 Macc 7,2 Maccabees 7,2 Macc 7
Reference,1 Esdras 4:35-40,1 Esdras 4:35–40,1 Esd 4:35–40
Book,Second Corinthians,2 Corinthians,2 Cor


## Sort and compare

References have comparison functions and can be sorted and combined.


In [5]:
const matt4 = __.r('Matt 4');
const luke7 = __.r('Luke 7');

console.log(matt4.lt(luke7) ? 'Matt 4 < Luke 7' : 'XXXX');
console.log(matt4.equals(luke7) ? 'XXXX' : 'Matt 4 != Luke 7');


Matt 4 < Luke 7
Matt 4 != Luke 7


In [6]:
const normal_refs = matches.map(([match, ref]) => ref).filter((ref) => ref !== null && !ref.isBook());
const sorted_refs = normal_refs.toSorted((a, b) => a.lt(b) ? -1 : a.equals(b) ? 0 : 1);

const data = sorted_refs.map((ref, i) => [i + 1, __.name(ref)]);

const html = table(data);

Deno.jupyter.html`${html}`;

1,Psalm 119:105–112
2,Amos 4:1
3,Wisdom of Solomon 7:21–30
4,2 Maccabees 7
5,1 Esdras 4:35–40
6,"Romans 12:1–2, 9–12"
7,Romans 12:9–21
8,2 Corinthians 4:16–5:5
9,2 Corinthians 5:11–15
10,2 Corinthians 5:11–15
11,2 Corinthians 5:11–21


## Collate References

In [7]:
const indent = "    ";
__.collateChapterReferences(sorted_refs).forEach(([library, book_collation]) => {
    console.log(library.name)
    book_collation.forEach(([book, chapter_collation]) => {
        console.log(indent + book.name);
        chapter_collation.forEach(([_, chapter_references]) => {
            console.log(indent + indent + __.numbers(__.sortReferences(chapter_references)));
        });
    });    
});

Old Testament
    Psalm
        119:105–112
    Amos
        4:1
Deuterocanonical
    Wisdom of Solomon
        7:21–30
    2 Maccabees
        7
Deuterocanonical (Orthodox)
    1 Esdras
        4:35–40
New Testament
    Romans
        12:1–2, 9–12, 9–21
    2 Corinthians
        4:16–5:5
        5:11–15, 11–15, 11–21, 16–21
    Philemon
        4–7


## Generate an index

There are shortcuts for common indexing tasks.

In [8]:
console.log(__.makeIndex(normal_refs));
console.log(__.makeSummary(normal_refs));  // <-- merge overlapping and combine adjacent references 
console.log(__.makeHotspots(normal_refs));

Ps 119:105–112, Amos 4:1, Wis 7:21–30, 2 Macc 7, 1 Esd 4:35–40, Rom 12:1–2, 9–12, 9–21, 2 Cor 4:16–5:5; 5:11–15, 11–15, 11–21, 16–21, Phlm 4–7
Ps 119:105–112, Amos 4:1, Wis 7:21–30, 2 Macc 7, 1 Esd 4:35–40, Rom 12:1–2, 9–21, 2 Cor 4:16–5:5; 5:11–21, Phlm 4–7
2 Cor 5, Rom 12


## Generate HTML links

To link references to online Bibles, just use __.template() with the template variables listed in README.md.

In [9]:
const bible_gateway = (
    '<a href="https://www.biblegateway.com/passage/'
  + '?search={LINK}&version=NRSVUE;WLC;VULGATE;SBLGNT">'
  + '{ABBREV_NAME}'
  + '</a>'
)

const ref = __.r('Prov 18:17')
const html = __.template(ref, bible_gateway)

Deno.jupyter.html`${html}`

Prov 18:17

## Combine reference links

Often we'll want to link more than one reference at a time. We can paginate or simply combine:

In [10]:
const refs = __.combineReferences(sorted_refs)
const html = __.template(refs, bible_gateway)

Deno.jupyter.html`${html}`

Ps 119:105–112; Amos 4:1; Wis 7:21–30; 2 Macc 7; 1 Esd 4:35–40; Rom 12:1–2, 9–21; 2 Cor 4:16–5:5; 5:11–21; Phlm 4–7

## Replace references with HTML links

In [11]:
var strs = [], tags = [];

__.findReferences(text, true, true).forEach(([match_str, ref]) => {
    strs.push(match_str)
    if (ref === null) { 
        tags.push(`<span style="color: red;">${match_str}</span>`);
    } else if (ref.isBook()) {
        tags.push(`<span style="color: green; font-weight: bold;">${match_str}</span>`);
    } else {
        const link = __.template(ref, bible_gateway);
        tags.push(`<b>${match_str}</b> <sup>[${link}]</sup>`);
    }
});

const html = __.sequentialReplace(text, strs, tags);

Deno.jupyter.html`<output style=\"font-size: larger;\">${html}</output>"`;

Human-written Bible references look like Rom 12.1-2, 9-12 [ Rom 12:1–2, 9–12 ] , 2 Cor. 4:16-5:5 [ 2 Cor 4:16–5:5 ] ,
or Philemon 4-7 [ Phlm 4–7 ] . They wrap lines, use spaces inconsistently, use colons and
periods interchangeably, indicate abbreviations with periods (or not), and
have commas both between and within references. They are sometimes
malformed, like Mt 1.10000 or 1:3-2 , but might also use number
abbreviations like Ps 119:105-12 [ Ps 119:105–112 ] . They might refer to Deuterocanonical
books (DC), like Wis 7:21-30 [ Wis 7:21–30 ] or 2 Macc 7 [ 2 Macc 7 ] , or Anagignoskomena (DCO), like 1
Esdras 4:35-40 [ 1 Esd 4:35–40 ] . A book name, like Second Corinthians , may provide context
for references that follow, such as 5:11-15 [ 2 Cor 5:11–15 ] or vv. 16-21 [ 2 Cor 5:16–21 ] . We want to match,
say, ' John ' but not match 'John Smith' in these cases. If we cite Romans 
(but then add a reference to 2ndCo 5:11-21 [ 2 Cor 5:11–21 ] in parentheses), a subsequent
reference like 12:9-21 [ Rom 12:9–21 ] should still be to Romans . Using letters for partial
verses, as in II Cor 5:11a-15d [ 2 Cor 5:11–15 ] , has no consistent meaning, so the letters
are ignored. Book aliases that are common words will be ignored except in
references, e.g. Am 4:1 [ Amos 4:1 ] , but not Am. "